# 三大法人現貨買賣金額對 0050 未來報酬影響
流程：GitHub → Colab → 測試／執行 → Google Drive。

In [ ]:
REPO_URL = 'https://github.com/hh4832/-institutional-spot-flow-study.git'
BRANCH = 'feature/phase2-flow-mechanism'
PROJECT_DIR = '/content/institutional-spot-flow-study'

In [ ]:
import os, subprocess
from pathlib import Path
project = Path(PROJECT_DIR)
if not project.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, PROJECT_DIR], check=True)
else:
    status = subprocess.run(['git', 'status', '--porcelain'], cwd=project, capture_output=True, text=True, check=True).stdout
    if status.strip():
        raise RuntimeError('工作目錄有未提交變更，請先檢查 git status；程式不會自動覆蓋。')
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=project, check=True)
    local_branch = subprocess.run(['git', 'show-ref', '--verify', '--quiet', f'refs/heads/{BRANCH}'], cwd=project).returncode == 0
    if local_branch:
        subprocess.run(['git', 'checkout', BRANCH], cwd=project, check=True)
        subprocess.run(['git', 'merge', '--ff-only', f'origin/{BRANCH}'], cwd=project, check=True)
    else:
        subprocess.run(['git', 'checkout', '-b', BRANCH, '--track', f'origin/{BRANCH}'], cwd=project, check=True)
os.chdir(project)

In [ ]:
!python -m pip install -q -r requirements.txt
!pytest -q

## FinLab 登入與真實資料研究
優先使用 Colab Secrets 的 `FINLAB_API_TOKEN`；若未設定，再以隱藏輸入取得。

In [ ]:
import getpass, finlab
try:
    from google.colab import userdata
    token = userdata.get('FINLAB_API_TOKEN')
except Exception:
    token = None
if not token:
    token = getpass.getpass('請輸入 FinLab API Token：')
finlab.login(token)

In [ ]:
from config import StudyConfig
from data_loader import load_finlab_data
from phase2_pipeline import run_phase2_study
print('[1/8] 下載 FinLab 資料')
raw = load_finlab_data('0050')
output_dir = run_phase2_study(raw, StudyConfig(study_mode='phase2_flow_mechanism'))
print(output_dir.resolve())

In [ ]:
from google.colab import drive
from datetime import datetime
import shutil
drive.mount('/content/drive')
destination_root = Path('/content/drive/MyDrive/Quant_Research/institutional-spot-flow-study/phase2_outputs')
destination_root.mkdir(parents=True, exist_ok=True)
destination = destination_root / output_dir.name
if destination.exists():
    raise FileExistsError(f'目的資料夾已存在，不覆蓋：{destination}')
shutil.copytree(output_dir, destination)
print(f'已複製至：{destination}')